# Réseaux de neurones - solutions

Le but de cet exercice est de vous donner l'experience d'écrire votre propre modèle et résoudre un problème vous-mêmes avec PyTorch.

Ceci est la solution.

In [ ]:
%%capture
!uv pip install -r requirements.txt

import torch
from torch import nn
import torchaudio

import matplotlib.pyplot as plt
from IPython.display import Audio
import numpy as np
import deeplake
import random
import seaborn as sn
import os
from utils import AudioDataset, collate_fn, compute_accuracy_and_conf_mat, BreastCancerDataset
import tqdm
import warnings

dataset = BreastCancerDataset()

In [ ]:
# on sépare en les données pour entraînement/validation/test
n = len(dataset)
print(n)
train_size = 350
val_size = 100
test_size = n - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

batch_size = 32 # vous pouvez essayer de jouer avec cette valeur
max_epochs = 10
num_workers = 4 # Vous pouvez en mettre plus ou moins selon la puissance de votre ordinateur

# dataloaders
train_loader = torch.utils.data.DataLoader(train_ds,
                          batch_size=batch_size, 
                          shuffle=True, 
                          num_workers=num_workers)
val_loader   = torch.utils.data.DataLoader(val_ds,
                          batch_size=batch_size, 
                          shuffle=False, 
                          num_workers=num_workers)
test_loader  = torch.utils.data.DataLoader(test_ds,
                          batch_size=batch_size, 
                          shuffle=False, 
                          num_workers=num_workers)

Le premier exercice est de compléter la classe ManualAudioNet. J'ai défini tous les layers, mais vous devez implémenter la "forward propagation". J'ai donné un exemple avec la première couche de convolution.

In [ ]:
class CancerMLP(nn.Module):
    def __init__(self, input_dim=30, n_classes=2, dropout_probability=0.2):
        super().__init__()
        self.input_dim = input_dim
        self.dropout_probability = dropout_probability
        self.n_classes = n_classes

        # repeating layers
        self.relu = nn.ReLU()
        self.drop_layer =  nn.Dropout(self.dropout_probability)

        # linear layers
        self.lin1 = nn.Linear(self.input_dim, 64)

        self.lin2 = nn.Linear(64, 128)

        self.lin3 = nn.Linear(128, 64)

        self.lin4 = nn.Linear(64, self.n_classes)

    def forward(self, input):
        # Première couche : linear + relu + dropout
        out = self.lin1(input)
        out = self.relu(out)
        out = self.drop_layer(out)

        # Deuxième couche : linear + relu + dropout
        out = self.lin2(out)
        out = self.relu(out)
        out = self.drop_layer(out)

        # Troisième couche : linear + relu + dropout
        out = self.lin3(out)
        out = self.relu(out)
        out = self.drop_layer(out)

        # Quatrième couche : linear + retourner la sortie
        out = self.lin4(out)
        return out


Je vous laisse maintenant tester si la sortie de votre réseau de neurone est correcte.

In [ ]:
test_model = CancerMLP()
test_model.eval()

test_features, test_target = train_ds[0]
with torch.no_grad():
    out = test_model(test_features)
assert out.shape == torch.Size([2]), print(f"La sortie est de taille {out.shape}, alors qu'elle devrait être de taille {torch.Size([2])}")
print("Test réussi!")

Maintenant que le modèle est implémenté, on doit vérifier s'il est possible de trouver un GPU au lieu d'utiliser un CPU.

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device("cpu")

print(f"Using {device} device")

On définit le modèle, la perte d'entraînement et notre optimiseur. J'ai décidé d'utiliser [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html), car c'est un optimiseur bien connu, mais on aurait pu utiliser SGD ou autre.

La perte que l'on utilise est la Cross Entropy Loss :

$$\ell(x,y) = -\log \frac{\exp\big(MLP(x)_{y}\big)}{\sum_{c=1}^C \exp\big(MLP(x)_{c}\big)} $$

Avec $C$ le nombre de classes et $MLP(x)_c$ la prédiction pour la $c$-ième classe.

In [ ]:
# on crée le modèle puis on le met sur le gpu
model = CancerMLP().to(device)

# set the loss function
criterion = torch.nn.CrossEntropyLoss()

# set the optimizer
# Vous pouvez jouer avec le learning rate, j'ai trouvé que 0.005 fonctionnait plutôt bien
optimizer = torch.optim.Adam(model.parameters(), lr=0.015)

Vous devez maintenant implémenter l'entraînement du modèle. Les instructions sont mises dans le code, vous devez simplement finir chaque ligne.



In [ ]:
for epoch in range(max_epochs):  # loop over the dataset multiple times    
    running_loss = 0.0
    with tqdm.tqdm(total=len(train_loader), desc="Epoch %s" % str(epoch+1)) as pbar:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore") 
            for batch in train_loader:
        
                inputs, targets = batch

                # Envoyer les données sur le GPU
                inputs = inputs.to(device)
                targets = targets.to(device)
        
                # Mettre à zéro les gradients
                optimizer.zero_grad()
        
                # Phase d'inférence
                outputs = model.forward(inputs)

                # calculer la perte
                loss = criterion(outputs, targets)

                # calculer les gradients grâce à la backprop
                loss.backward()

                #Faire la mise à jour
                optimizer.step()
        
                # print statistics
                running_loss += loss.item()
                pbar.update(1)
            pbar.set_postfix({'loss': f"{running_loss/len(train_ds):.5f}"})

Finalement, on va tester le modèle sur les données d'entraînement. La majorité du code est pareil, mais cette fois ci on va calculer l'accuracy au lieu de la cross entropy.

La formule de l'accuracy est la suivante :

$$Acc = \frac{1}{n}\sum_{i=1}^n \begin{cases} 1 & \text{si } \widehat{y}_i = y_i \\ 0 & \text{si } \widehat{y}_i \neq y_i\end{cases} $$

pour un jeu de données $S = \{(x_i, y_i)\}_{i=1}^n$ et $\widehat{y}_i$ la classe prédite par $MLP(x_i)$.


Puisqu'on veut calculer l'accuracy sur le jeu de données et non sur chaque batch, calculez simplement le nombre de données bien prédites ($n \cdot Acc$) et le code va automatiquement diviser par la taille du jeu de données.

In [ ]:
# mettre le modèle en mode d'évaluation, car la batch norm et le dropout
# ne fonctionne pas pareil en entraînement et en évaluation
model.eval()

# envoyer le modèle sur le GPU
model.to(device)

acc = 0
# On ne veut pas calculer les gradients, ils ne nous serviront pas.
with torch.no_grad():
    with tqdm.tqdm(total=len(test_loader), desc="Computing test error") as pbar:
        for sample in test_loader:
        
            inputs, targets = sample

            # Envoyer les données sur le GPU
            inputs = inputs.to(device)
            targets = targets.to(device)

            # Inférence du modèle
            outputs = model(inputs)

            # Trouver la classe prédite par le modèle
            pred = torch.argmax(outputs, dim=1)

            # Calculer le nombre de données prédites correctement
            acc_batch = (pred == targets).sum().item()
            
            acc += acc_batch
            pbar.update(1)

model.cpu()

acc = acc / len(test_ds)
print(f"L'accuracy sur le jeu de données de test est de {100*acc:.2f} %")

On calcule maintenant les matrices de confusion pour ce modèle.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20,5))

acc, conf_mat = compute_accuracy_and_conf_mat(model, train_loader, device)
acc = acc / len(train_ds)
sn.heatmap(conf_mat.to(int), annot=True, fmt="d", ax=axs[0])
axs[0].set_title("Matrice de confusion du jeu de donnés d'entraînement")
print(f"L'accuracy sur le jeu de données d'entraînement est de {100*acc:.2f} %")

acc, conf_mat = compute_accuracy_and_conf_mat(model, val_loader, device)
acc = acc / len(val_ds)
sn.heatmap(conf_mat.to(int), annot=True, fmt="d", ax=axs[1])
axs[1].set_title("Matrice de confusion du jeu de donnés de validation")
print(f"L'accuracy sur le jeu de données de validation est de {100*acc:.2f} %")

acc, conf_mat = compute_accuracy_and_conf_mat(model, test_loader, device)
acc = acc / len(test_ds)
sn.heatmap(conf_mat.to(int), annot=True, fmt="d", ax=axs[2])
axs[2].set_title("Matrice de confusion du jeu de donnés de test")
print(f"L'accuracy sur le jeu de données de test est de {100*acc:.2f} %")